In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ==============================
# SIMULATION SETTINGS
# ==============================

n = 500
p = 5000
d = p + 1

m_true = 20
n_runs = 50

beta0 = 3.0
rho = 0.8

sigma_true = 3.0
sigma2_true = sigma_true**2


beta_seed = 1712


# Dataset file seed_00.csv uses RNG seed 500,
# seed_01.csv uses RNG seed 501,
# ...
# seed_49.csv uses RNG seed 549.
data_seed_offset = 500


output_dir = Path("data_5000")

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ==============================
# FIXED ACTIVE SET AND BETA
# ==============================

beta_rng = np.random.default_rng(
    beta_seed
)


# Select 20 distinct active slope predictors.
true_active = beta_rng.choice(
    p,
    size=m_true,
    replace=False,
)

true_active.sort()


# Generate the 20 nonzero coefficients once.
# They remain fixed across all 50 datasets.
beta_slopes = np.zeros(
    p,
    dtype=np.float64,
)

beta_slopes[
    true_active
] = beta_rng.normal(
    loc=0.0,
    scale=1.0,
    size=m_true,
)


# Intercept followed by the 5,000 slope coefficients.
beta_full = np.concatenate(
    (
        [beta0],
        beta_slopes,
    )
)


# ==============================
# SAVE SHARED TRUE MODEL
# ==============================

is_true_active = np.zeros(
    p,
    dtype=np.int64,
)

is_true_active[
    true_active
] = 1


true_model_df = pd.DataFrame(
    {
        "design_column_0_based":
            np.arange(
                d
            ),

        "variable_name":
            (
                ["Intercept"]
                + [
                    f"X{j}"
                    for j in range(
                        1,
                        p + 1,
                    )
                ]
            ),

        "is_intercept":
            np.concatenate(
                (
                    [1],
                    np.zeros(
                        p,
                        dtype=np.int64,
                    ),
                )
            ),

        "is_true_active":
            np.concatenate(
                (
                    [0],
                    is_true_active,
                )
            ),

        "beta":
            beta_full,
    }
)


true_model_file = (
    output_dir
    / "fixed_beta_and_active_set.csv"
)


true_model_df.to_csv(
    true_model_file,
    index=False,
    float_format="%.17g",
)


print(
    "Fixed active slope variables, "
    "using X1,...,X5000:"
)

print(
    true_active + 1
)


print(
    "\nFixed nonzero coefficients:"
)

print(
    true_model_df.loc[
        true_model_df[
            "is_true_active"
        ] == 1,
        [
            "variable_name",
            "beta",
        ],
    ].to_string(
        index=False
    )
)


# ==============================
# GENERATE 50 DATASETS
# ==============================

scale = np.sqrt(
    1.0 - rho**2
)


for seed in range(
    n_runs
):


    data_rng_seed = (
        data_seed_offset
        + seed
    )


    rng = np.random.default_rng(
        data_rng_seed
    )


    # --------------------------
    # Raw Gaussian AR(1) design
    # --------------------------

    X_raw = np.empty(
        (n, p),
        dtype=np.float64,
    )


    # First predictor ~ N(0,1).
    X_raw[:, 0] = rng.normal(
        loc=0.0,
        scale=1.0,
        size=n,
    )


    # Remaining predictors follow AR(1):
    #
    # X_j = rho * X_{j-1}
    #       + sqrt(1-rho^2) * epsilon_j
    #
    for j in range(
        1,
        p,
    ):

        X_raw[:, j] = (
            rho
            * X_raw[:, j - 1]

            + scale
            * rng.normal(
                loc=0.0,
                scale=1.0,
                size=n,
            )
        )


    # --------------------------
    # Standardise slopes only
    # --------------------------

    X_mean = X_raw.mean(
        axis=0
    )


    X_sd = X_raw.std(
        axis=0,
        ddof=1,
    )


    if np.any(
        X_sd == 0.0
    ):

        raise ValueError(
            f"Seed {seed} "
            f"(RNG seed {data_rng_seed}) "
            "generated a constant predictor."
        )


    X = (
        X_raw
        - X_mean[None, :]
    ) / X_sd[None, :]


    # --------------------------
    # Add untouched intercept
    # --------------------------

    X_full = np.column_stack(
        (
            np.ones(
                n,
                dtype=np.float64,
            ),
            X,
        )
    )


    assert (
        X_full.shape
        == (n, d)
    )


    assert np.all(
        X_full[:, 0]
        == 1.0
    )


    # --------------------------
    # Generate noise and y
    # --------------------------

    noise = rng.normal(
        loc=0.0,
        scale=sigma_true,
        size=n,
    )


    y = (
        X_full
        @ beta_full
        + noise
    )


    # --------------------------
    # Save dataset CSV
    # --------------------------

    dataset_df = pd.DataFrame(
        X_full,
        columns=(
            ["Intercept"]
            + [
                f"X{j}"
                for j in range(
                    1,
                    p + 1,
                )
            ]
        ),
    )


    dataset_df[
        "y"
    ] = y


    dataset_df[
        "noise"
    ] = noise


    dataset_file = (
        output_dir
        / f"seed_{seed:02d}.csv"
    )


    dataset_df.to_csv(
        dataset_file,
        index=False,
        float_format="%.17g",
    )


    print(
        f"Saved seed {seed:02d}: "
        f"{dataset_file.name} "
        f"(RNG seed = {data_rng_seed})"
    )


# ==============================
# FINAL CHECKS
# ==============================

generated_files = sorted(
    output_dir.glob(
        "seed_*.csv"
    )
)


if (
    len(
        generated_files
    )
    != n_runs
):

    raise ValueError(
        f"Expected {n_runs} dataset files, "
        f"but found {len(generated_files)}."
    )


if (
    int(
        true_model_df[
            "is_true_active"
        ].sum()
    )
    != m_true
):

    raise ValueError(
        "Incorrect number of active predictors "
        "in the saved true-model file."
    )


# Verify dimensions of one generated file.
check_df = pd.read_csv(
    generated_files[0]
)


expected_columns = (
    ["Intercept"]
    + [
        f"X{j}"
        for j in range(
            1,
            p + 1,
        )
    ]
    + [
        "y",
        "noise",
    ]
)


if (
    list(
        check_df.columns
    )
    != expected_columns
):

    raise ValueError(
        "Generated dataset columns "
        "do not match the expected format."
    )


if (
    len(
        check_df
    )
    != n
):

    raise ValueError(
        f"Expected n={n} rows, "
        f"but found {len(check_df)}."
    )


# ==============================
# FINAL SUMMARY
# ==============================

print(
    "\n"
    + "=" * 70
)

print(
    "GENERATION COMPLETE"
)

print(
    "=" * 70
)


print(
    f"\nOutput folder: "
    f"{output_dir.resolve()}"
)


print(
    f"Number of datasets: "
    f"{n_runs}"
)


print(
    f"Dataset files: "
    f"seed_00.csv to "
    f"seed_{n_runs - 1:02d}.csv"
)


print(
    f"True-model file: "
    f"{true_model_file.name}"
)


print(
    "\nSimulation setting:"
)

print(
    f"  n = {n}"
)

print(
    f"  p = {p}"
)

print(
    f"  active predictors = {m_true}"
)

print(
    f"  intercept = {beta0}"
)

print(
    f"  rho = {rho}"
)

print(
    f"  sigma = {sigma_true}"
)

print(
    f"  sigma^2 = {sigma2_true}"
)


print(
    "\nRandom seeds:"
)

print(
    f"  beta seed = "
    f"{beta_seed}"
)

print(
    f"  data RNG seeds = "
    f"{data_seed_offset} "
    f"to "
    f"{data_seed_offset + n_runs - 1}"
)


print(
    "\nFixed active variables:"
)

print(
    true_active + 1
)


print(
    "\nFixed active coefficients:"
)

print(
    true_model_df.loc[
        true_model_df[
            "is_true_active"
        ] == 1,
        [
            "variable_name",
            "beta",
        ],
    ].to_string(
        index=False
    )
)